# Track 2 · Stage 4 — Evaluation

## Run **GATE 3 first**, before any generation or training.

`whisper-large-v2` zero-shot on the test set should score **≈ 52.0 MER / 42.9 CBA-HE**.
Inference only — no training, ~3 GB, ~40 min on a T4.

That number is published in the paper, so reproducing it validates decoding,
normalization, word-level language ID, and both metrics end to end. It is **not**
our baseline — it is the calibration of the measuring instrument. A broken metric
makes every downstream result uninterpretable.

Then decode M6/M7/M8 and the `whisper-small` zero-shot baseline they are measured against.

### Decoding reproduces WhisperX, not a per-clip loop
The paper decodes whole recordings with WhisperX. We use **faster-whisper**, the engine
WhisperX wraps: language detected **once per recording**, 30-second chunks with real
context, beam 5, VAD, temperature fallback, and a compression-ratio threshold that aborts
repetition loops.

This matters enormously. Decoding the 3,136 isolated 6-second clips instead makes Whisper
render English loanwords in Devanagari, so the hypothesis contains **no script boundary and
no switch bigram can match**. Measured on real test audio with one model:

| decoding | %Latin in hyp | MER | CBA-HE |
|---|---|---|---|
| reference | 21.6% | – | – |
| per-clip | **0.0%** | 182.6 | 0.0 |
| recording-level | 12.2% | 103.1 | 1.6 |

`utt_id` is `<speaker>_<recording>_<index>`, so the `test` config already on the Hub is
regrouped into its 30 recordings here — no re-upload.

In [ ]:
!pip install -q git+https://github.com/BRUH-MAIN/codeswitching.git
!pip install -q -U transformers jiwer faster-whisper ctranslate2
!pip install -q "datasets<4" librosa soundfile   # see 02_train: torchcodec

import os, subprocess, sys
os.environ["HF_HOME"] = "/kaggle/temp/hf"

from kaggle_secrets import UserSecretsClient
# Put the token in the ENVIRONMENT, never in argv: a CLI arg lands verbatim in
# every traceback and in `ps` output.
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])

REAL = "RohanRamesh/mucs-he-cs"
OUT  = "/kaggle/working"

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    subprocess.run([sys.executable, "-m", *args], check=True)   # inherits HF_TOKEN

!nvidia-smi --query-gpu=name,memory.total --format=csv

## GATE 3 — calibrate the metric against a published number

Inference only. ~15 min on a T4 (30 recordings, not 3,136 clips).

In [ ]:
run("csasr.eval.decode", "--model", "openai/whisper-large-v2",
    "--engine", "faster-whisper", "--mode", "recording", "--language", "none",
    "--test-hf", REAL, "--test-config", "test",
    "--out", f"{OUT}/hyp_largev2_zeroshot.jsonl",
    "--refs-out", f"{OUT}/refs_recording.jsonl")

In [ ]:
from csasr.eval.score import score

for mode in ("word", "hybrid"):
    res = score(f"{OUT}/refs_recording.jsonl", f"{OUT}/hyp_largev2_zeroshot.jsonl", mer_mode=mode)
    print(f"MER mode={mode:<7} -> MER {res['mer']:.1f}   CBA-HE {res['cba_he']:.1f}   CBA-EH {res['cba_eh']:.1f}")

print("\npaper (large-v2 zero-shot): MER 52.0   CBA-HE 42.9   CBA-EH 36.x")
print("Whichever mode lands near 52.0 is the definition the authors used.")
print("\nIf MER is far from 52 or CBA-HE far from 42.9, STOP. Do not generate data")
print("against a ruler that does not reproduce a published number.")

In [ ]:
# Sanity: the hypotheses must carry BOTH scripts, or CBA is structurally zero.
from csasr.manifest import read_jsonl
from csasr.lid import Lang, count_words
from csasr.normalize import normalize

def mix(t):
    c = count_words(normalize(t, "scoring")); tot = sum(c.values()) or 1
    return c[Lang.HI] / tot, c[Lang.EN] / tot

refs = {r["utt_id"]: r["text"] for r in read_jsonl(f"{OUT}/refs_recording.jsonl")}
hyps = list(read_jsonl(f"{OUT}/hyp_largev2_zeroshot.jsonl"))
rh, re_ = mix(" ".join(refs.values()))
hh, he = mix(" ".join(h["hyp"] for h in hyps))
print(f"REFERENCE : {rh:5.1%} Devanagari  {re_:5.1%} Latin")
print(f"HYPOTHESIS: {hh:5.1%} Devanagari  {he:5.1%} Latin")
print("detected languages:", {h["detected_language"] for h in hyps})

## whisper-small zero-shot — the actual baseline for M6/M7/M8

In [ ]:
run("csasr.eval.decode", "--model", "openai/whisper-small",
    "--engine", "faster-whisper", "--mode", "recording", "--language", "none",
    "--test-hf", REAL, "--test-config", "test",
    "--out", f"{OUT}/hyp_small_zeroshot.jsonl")

## Decode the fine-tuned models

`decode.py` converts each HF checkpoint to CTranslate2 on first use and caches it, so
faster-whisper can load it. Run this **only after** `02_train.ipynb` has pushed the repos.

In [ ]:
for m in ("m6", "m7", "m8"):
    run("csasr.eval.decode", "--model", f"RohanRamesh/whisper-small-cs-{m}",
        "--engine", "faster-whisper", "--mode", "recording", "--language", "none",
        "--test-hf", REAL, "--test-config", "test",
        "--ct2-cache", "/kaggle/temp/ct2",
        "--out", f"{OUT}/hyp_{m}.jsonl")

## Results — reproduce the ordering of Table 2

In [ ]:
systems = [
    ("large-v2 zero-shot", "hyp_largev2_zeroshot.jsonl", 52.0),
    ("small zero-shot",    "hyp_small_zeroshot.jsonl",   None),
    ("M6 (T1, 8h)",        "hyp_m6.jsonl",               48.2),
    ("M7 (T2, 22h)",       "hyp_m7.jsonl",               40.8),
    ("M8 (T2 + mono)",     "hyp_m8.jsonl",               39.2),
]
rows = []
for name, f, paper_mer in systems:
    r = score(f"{OUT}/refs_recording.jsonl", f"{OUT}/{f}")
    rows.append((name, r, paper_mer))

print(f"{'system':<20}{'MER':>8}{'paper':>8}{'CBA-HE':>9}{'CBA-EH':>9}")
for name, r, pm in rows:
    p = f"{pm:.1f}" if pm else "-"
    print(f"{name:<20}{r['mer']:>8.1f}{p:>8}{r['cba_he']:>9.1f}{r['cba_eh']:>9.1f}")

mers = [r["mer"] for _, r, _ in rows[2:]]
print("\nM6 > M7 > M8 ordering reproduced:", mers == sorted(mers, reverse=True))

with open(f"{OUT}/table2.json", "w") as fh:
    json.dump([{"system": n, **r} for n, r, _ in rows], fh, indent=2)